# Feature Engineering with Feast

A major bottleneck in production ML is **training-serving skew** — the
discrepancy between how features are computed for training versus serving.
A feature store gives us a single feature definition used by *both* paths,
and **point-in-time joins** guarantee that a label row only ever sees
feature values known at its timestamp.

This project's deployment pattern (scheduled batch scoring) needs only the
**offline store**: the training set and the scoring input are both produced
by `get_historical_features` over the same parquet source — the feature
view below is declared `online=False` and nothing is ever materialized to
an online store.


## Environment

In [ ]:
import os

# Kernel env on PATH so %%bash cells can call feast/aws/docker directly; the
# aws CLI and Feast binaries live in the workspace venv (/opt/mlops-venv).
os.environ["PATH"] = "/opt/mlops-venv/bin:" + os.environ["PATH"]
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

# This project's own moto (see docker-compose.yml) — NOT the lab's shared one.
os.environ["MOTO_URL"] = "http://telco-customer-churn-moto:5000"

## The infrastructure as code

Everything this project needs from Docker is declared in its own
`docker-compose.yml` — a batch feature store needs no always-on services,
so the file is deliberately small: a single moto container serving S3/STS
for the SageMaker local-mode jobs.


In [ ]:
%%bash
cat ../docker-compose.yml

In [ ]:
%%bash
docker compose -f ../docker-compose.yml up -d

In [ ]:
%%bash
docker compose -f ../docker-compose.yml ps

Moto answers S3 on `telco-customer-churn-moto:5000` — the endpoint the
project's `configs/local.yaml` wires into the SageMaker session. Quick check:


In [ ]:
%%bash
aws s3 mb "s3://telco-customer-churn" --endpoint-url "$MOTO_URL" 2>/dev/null; aws s3 ls --endpoint-url "$MOTO_URL"

## Build the feature tables

`prepare_data.py` projects the raw CRM export into the feature contract:
entity key (`customer_id`), snapshot `event_timestamp` (tz-aware UTC — the
offline store's temporal filters need it), and 15 typed features.


In [ ]:
%%bash
cd .. && python -m telco_customer_churn.prepare_data && ls -la data/processed/

## The Feast repository

`feature_repo/` declares the source, entity, and feature view as code —
the same definitions Feast applies to a registry it can reproduce anywhere.


In [ ]:
%%bash
cat ../feature_repo/feature_store.yaml

In [ ]:
%%bash
cat ../feature_repo/data_sources.py ../feature_repo/entities.py

In [ ]:
%%bash
cat ../feature_repo/feature_views.py

## Apply the definitions

`feast apply` registers the repo into `feature_repo/data/registry.db` —
the artifact that makes `get_historical_features` reproducible.


In [ ]:
%%bash
cd ../feature_repo && feast apply

In [ ]:
%%bash
cd ../feature_repo && feast feature-views list

Note what did **not** happen: there is no `feast materialize` step. The view
is `online=False` — this project never serves point lookups, so there is
nothing to push to an online store. That is the whole point of an
offline-only deployment: less infrastructure, same feature contract.


## Point-in-time retrieval

The label table (`churn_labels.parquet`) carries `customer_id` +
`event_timestamp` + `churn`; `get_historical_features` joins each label row
to the latest feature values at-or-before its timestamp. The returned frame
is the training set — identical in shape to the scoring input the deploy
stage will request for the full population.


In [ ]:
import pandas as pd
from feast import FeatureStore

fs = FeatureStore(repo_path="../feature_repo")
labels = pd.read_parquet("../data/processed/churn_labels.parquet")
training_df = fs.get_historical_features(
    entity_df=labels.head(500),
    features=[
        "customer_subscription:tenure_months",
        "customer_subscription:month_to_month",
        "customer_subscription:monthly_charges",
        "customer_subscription:num_add_ons",
    ],
).to_df()
training_df.head()